## Module 4-3 Text Vectorization and Word Clouds

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

### 1. Load last class's R&D disclosures

`data/rnd_disclosures.csv` has one row per filing, with `RnD_Text` holding all of that filing's R&D-related sentences (one per line). For building a document-term matrix, we want one row per *sentence*, so we split each filing's text into individual sentences first.

In [ ]:
rnd_df = pd.read_csv('../data/rnd_disclosures.csv')

records = []
for _, row in rnd_df.iterrows():
    for sentence in str(row['RnD_Text']).split('\n'):
        sentence = sentence.strip()
        if len(sentence) > 0:
            records.append({'CIK': row['CIK'], 'sentence': sentence})

sentences_df = pd.DataFrame(records)
print(sentences_df.shape)
sentences_df.head()

### 2. Bag-of-Words with `CountVectorizer`

`CountVectorizer` turns a list of documents (here, sentences) into a **document-term matrix**: one row per document, one column per word in the vocabulary, and each cell is the number of times that word appears in that document.

In [ ]:
count_vectorizer = CountVectorizer(stop_words='english', min_df=2)
dtm = count_vectorizer.fit_transform(sentences_df['sentence'])

dtm_df = pd.DataFrame(
    dtm.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=sentences_df.index
)
print('Document-term matrix shape:', dtm_df.shape)
dtm_df.head()

In [ ]:
# Most frequent terms across all R&D sentences
term_counts = dtm_df.sum().sort_values(ascending=False)
term_counts.head(15)

### 3. From counts to TF-IDF

Raw counts favor common words. **TF-IDF** (term frequency - inverse document frequency) reweights each word by how *distinctive* it is: words that show up in almost every sentence get pushed down, even if they are frequent.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', min_df=2)
tfidf = tfidf_vectorizer.fit_transform(sentences_df['sentence'])

tfidf_df = pd.DataFrame(
    tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=sentences_df.index
)

term_tfidf = tfidf_df.sum().sort_values(ascending=False)

comparison = pd.DataFrame({
    'top_by_count': term_counts.index[:15],
    'top_by_tfidf': term_tfidf.index[:15],
})
comparison

### 4. Capturing phrases with n-grams

A single word ("clinical", "intellectual") often loses meaning on its own. Setting `ngram_range=(1, 2)` also keeps two-word phrases ("clinical trials", "intellectual property") as vocabulary terms.

In [ ]:
bigram_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
bigram_tfidf = bigram_vectorizer.fit_transform(sentences_df['sentence'])
bigram_df = pd.DataFrame(bigram_tfidf.toarray(), columns=bigram_vectorizer.get_feature_names_out())

top_terms = bigram_df.sum().sort_values(ascending=False)
top_phrases = top_terms[top_terms.index.str.contains(' ')]  # keep two-word phrases only
top_phrases.head(15)

### 5. Word clouds

A word cloud is just a picture of a term-frequency table: bigger word = higher weight. We can feed it either the raw counts or the TF-IDF weights we already built above.

In [ ]:
count_freqs = term_counts.to_dict()
wc_count = WordCloud(width=900, height=450, background_color='white').generate_from_frequencies(count_freqs)

plt.figure(figsize=(10, 5))
plt.imshow(wc_count, interpolation='bilinear')
plt.axis('off')
plt.title('R&D sentences -- raw term counts')
plt.show()

In [ ]:
tfidf_freqs = term_tfidf.to_dict()
wc_tfidf = WordCloud(width=900, height=450, background_color='white').generate_from_frequencies(tfidf_freqs)

plt.figure(figsize=(10, 5))
plt.imshow(wc_tfidf, interpolation='bilinear')
plt.axis('off')
plt.title('R&D sentences -- TF-IDF weighted')
plt.show()

### 6. Comparing two filings

We only have two filings with R&D-related sentences. Rather than pooling them, we can build one word cloud per filing (using `WordCloud.generate()`, which handles its own tokenization and stopword removal from raw text) to see how the two firms' R&D disclosures differ.

In [ ]:
fig, axes = plt.subplots(1, len(rnd_df), figsize=(14, 5))
if len(rnd_df) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, rnd_df.iterrows()):
    wc = WordCloud(width=800, height=450, background_color='white', stopwords=STOPWORDS).generate(str(row['RnD_Text']))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f"CIK {row['CIK']}")

plt.tight_layout()
plt.show()

Notice that CIK 1527728's cloud reads like genuine R&D discussion (*clinical trials, patents, regulatory approval*), while CIK 1647822's cloud is dominated by financial-statement boilerplate (*Total, Net income, liabilities, Balance*). Our keyword matcher in 4-2 flagged some sentences that merely mention a trigger word near an unrelated balance-sheet line item -- a real limitation of keyword-based extraction that we'll revisit when we compare against FinBERT's context-aware classification next.

### Next class

Count vectors and TF-IDF vectors are also exactly what feeds classic machine learning text classifiers (Naive Bayes, SVM, random forests). In the next notebook, instead of counting or reweighting words ourselves, we use **FinBERT** -- a large language model pretrained on financial text -- to read each sentence in context and classify its sentiment directly.